# Taller de NLP: feedback y sentimiento

Analizaremos reseñas de hoteles en español con **spaCy**, **TextBlob**, **Gensim** y **Pandas**.

## 1. Preparar el entorno

Instalamos las librerías y el modelo grande de español de spaCy.

In [ ]:
!pip -q install spacy textblob gensim pandas matplotlib
!python -m spacy download es_core_news_lg

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import spacy
from textblob import TextBlob
from gensim import corpora
from gensim.models import LdaModel

## 2. Descargar y cargar los datos

Usamos el archivo balanceado del dataset [Andalusian Hotels’ Reviews](https://www.kaggle.com/datasets/chizhikchi/andalusian-hotels-reviews-unbalanced) de Kaggle. Lo guardamos como `feedback_clientes.csv`.

In [ ]:
!wget -q https://www.kaggle.com/api/v1/datasets/download/chizhikchi/andalusian-hotels-reviews-unbalanced -O hotel_reviews.zip
!unzip -qo hotel_reviews.zip
!cp Balanced_AHR.csv feedback_clientes.csv

data = pd.read_csv('feedback_clientes.csv')
data = data[['review_text', 'rating']].dropna().copy()
data['rating'] = pd.to_numeric(data['rating'])

print('Cantidad de comentarios:', len(data))
data.head()

## 3. Limpiar y lematizar

Convertimos el texto a minúsculas, eliminamos stopwords y signos, y conservamos el lema de cada palabra.

In [ ]:
nlp = spacy.load('es_core_news_lg')
documents = list(nlp.pipe(data['review_text'], batch_size=64))

# Lematizamos y quitamos palabras vacías, números y signos.
data['clean_text'] = [
    ' '.join(
        token.lemma_.lower()
        for token in document
        if token.is_alpha and not token.is_stop
    )
    for document in documents
]

data[['review_text', 'clean_text']].head()

## 4. Clasificar el sentimiento

TextBlob calcula una polaridad exploratoria. Como su analizador predeterminado funciona mejor en inglés, usamos la calificación del cliente para la categoría final: 1–2 negativa, 3 neutral y 4–5 positiva.

In [ ]:
# Calculamos una polaridad exploratoria con TextBlob.
data['polarity'] = [
    TextBlob(text).sentiment.polarity
    for text in data['review_text']
]

def classify_sentiment(rating):
    if rating >= 4:
        return 'Positivo'
    if rating <= 2:
        return 'Negativo'
    return 'Neutral'

data['sentiment'] = data['rating'].apply(classify_sentiment)
data[['review_text', 'rating', 'polarity', 'sentiment']].head()

## 5. Distribución porcentual

El gráfico resume la salud general del feedback.

In [ ]:
sentiment_order = ['Positivo', 'Neutral', 'Negativo']
sentiment_percent = (
    data['sentiment']
    .value_counts(normalize=True)
    .reindex(sentiment_order, fill_value=0)
    .mul(100)
)

ax = sentiment_percent.plot(
    kind='bar',
    color=['#2E8B57', '#E0A800', '#C94C4C'],
    figsize=(8, 5)
)
ax.set_title('Distribución porcentual del sentimiento')
ax.set_xlabel('Sentimiento')
ax.set_ylabel('Porcentaje de comentarios (%)')
ax.tick_params(axis='x', rotation=0)

for index, value in enumerate(sentiment_percent):
    ax.text(index, value + 0.5, f'{value:.1f}%', ha='center')

plt.tight_layout()
plt.show()
sentiment_percent.round(1)

## 6. Similitud con un comentario ideal

Comparamos cada reseña con una experiencia ideal y mostramos los cinco registros más cercanos.

In [ ]:
ideal_comment = (
    'La habitación estaba limpia y tranquila, el personal fue amable '
    'y el servicio del hotel fue excelente.'
)
ideal_document = nlp(ideal_comment)

# Usamos los vectores del modelo grande de spaCy.
data['similarity_to_ideal'] = [
    document.similarity(ideal_document)
    for document in documents
]

data.nlargest(5, 'similarity_to_ideal')[
    ['review_text', 'rating', 'similarity_to_ideal']
]

## 7. LDA sobre comentarios negativos

Entrenamos tres tópicos con las reseñas negativas. Los nombres analíticos se asignan según las palabras clave más representativas.

In [ ]:
negative_texts = data.loc[
    data['sentiment'] == 'Negativo', 'clean_text'
]
negative_tokens = [text.split() for text in negative_texts if text.strip()]

dictionary = corpora.Dictionary(negative_tokens)
dictionary.filter_extremes(no_below=5, no_above=0.6)
corpus = [dictionary.doc2bow(tokens) for tokens in negative_tokens]

lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=3,
    passes=10,
    random_state=42
)

topic_names = {
    0: 'Atención y recepción',
    1: 'Habitaciones, limpieza y descanso',
    2: 'Comida e instalaciones',
}

for topic_id, words in lda_model.show_topics(
    num_topics=3, num_words=8, formatted=False
):
    keywords = ', '.join(word for word, weight in words)
    print(f'Tema {topic_id + 1} - {topic_names[topic_id]}: {keywords}')

## 8. Conclusiones

El análisis indica que los hoteles deben priorizar tres áreas: **atención y recepción**, **habitaciones, limpieza y descanso**, y **comida e instalaciones**. Conviene responder con rapidez a los problemas, mejorar el mantenimiento y la limpieza, controlar los ruidos nocturnos y revisar la calidad de la comida y la piscina. Los comentarios más cercanos al ideal muestran que los clientes valoran especialmente una habitación limpia y tranquila, acompañada de un trato amable. Estas acciones pueden reducir las reseñas negativas y mejorar la experiencia general.